# NeuralGCM lead-time **ensemble** sweep — Aug 2025 St. John's heat wave

The deterministic sweep gives one forecast per lead, so a long-lead *miss* could be
luck rather than true unpredictability. This notebook runs an **N-member ensemble per
lead** and measures the robust signal: the **hit rate** — the fraction of members that
capture the heat wave — as a function of lead time. The crossover (hit rate climbing
from ~0 to ~1) is the defensible "best initialisation window".

**Event — St. John's Intl A (ECCC 8403505) observed daily Tmax (°C):**
Aug 9: 30.5 · 10: 27.7 · 11: 28.1 · **12: 30.9 (peak)** · 13: 30.3 · 14: 27.8 · 15: 29.2.

### ⚠️ Requires a STOCHASTIC checkpoint
NeuralGCM ensemble spread comes from the stochastic model component, seeded by the
`encode` rng key. With a **deterministic** checkpoint the key is inert and every member
is identical (zero spread). This notebook defaults to `v1/stochastic_1_4_deg.pkl` and
checks the spread at the end — heed the warning if it fires.

### Before you Run all
1. **Runtime ▸ Change runtime type ▸ GPU** (High-RAM if available).
2. Run all. ERA5 is loaded once; **members are batched with `jax.vmap`**, so each lead is
   a single parallel GPU call — the whole sweep is minutes, not hours (and finishes well
   before Colab idle-disconnects). Start with `N_MEMBERS=8`, daily cadence. If the GPU
   OOMs on the batched call, lower `N_MEMBERS` or use the 2.8° checkpoint.
3. The stochastic 1.4° model is finer than the deterministic 2.8° sweep — for a
   grid-matched comparison instead use `v1_precip/stochastic_precip_2_8_deg.pkl`.

> **Still open (not fixed here):** the hit rate is scored against ERA5 *on the model grid*,
> which at this resolution is a cool, partly-oceanic T₁₀₀₀ (~20 °C) — far below the observed
> 30.9 °C. So a high hit rate means "tracks the coarse-grid ERA5 signal", **not** "reproduces
> the 30.9 °C heat wave". Anomaly-space scoring is the fix — ask for that variant.

In [ ]:
!pip install -q -U neuralgcm dinosaur gcsfs

In [ ]:
import gc, jax
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import gcsfs, pickle, neuralgcm
from dinosaur import horizontal_interpolation, spherical_harmonic, xarray_utils

print("JAX devices:", jax.devices())
if not any(d.platform in ("gpu", "cuda") for d in jax.devices()):
    print("\u26a0\ufe0f  No GPU detected \u2014 Runtime \u25b8 Change runtime type \u25b8 GPU, then Run all.")

# Non-LaTeX style (the repo's common.py uses usetex, which Colab usually lacks).
plt.rcParams.update({"figure.dpi": 120, "font.size": 10, "axes.grid": True,
                     "grid.alpha": 0.3, "axes.titlesize": 11})

## Config

In [ ]:
# REQUIRES A STOCHASTIC CHECKPOINT (see note at top).
MODEL_NAME = "v1/stochastic_1_4_deg.pkl"            # canonical stochastic ensemble model
# Grid-matched to the deterministic 2.8° sweep (cheaper, more GPU headroom):
#   MODEL_NAME = "v1_precip/stochastic_precip_2_8_deg.pkl"
ERA5_PATH = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

# Observed St. John's Intl A daily Tmax (°C).
OBS_TMAX = {"2025-08-09": 30.5, "2025-08-10": 27.7, "2025-08-11": 28.1,
            "2025-08-12": 30.9, "2025-08-13": 30.3, "2025-08-14": 27.8,
            "2025-08-15": 29.2}
EVENT_PEAK  = np.datetime64("2025-08-12")
EVENT_START = np.datetime64("2025-08-09")
EVENT_END   = np.datetime64("2025-08-15")

LEAD_TIMES_DAYS = [20, 18, 16, 14, 12, 10, 9, 7, 5, 3]

N_MEMBERS    = 8      # ensemble members per lead
MEMBER_BATCH = 2      # members per vmap GPU call. Peak GPU mem ~ this many
                      # trajectories. Lower to 1 if OOM; raise (e.g. 4) if you have RAM
                      # or use the 2.8° checkpoint (much smaller, can take 8).
CAPTURE_TOL  = 2.0    # °C; a member "captures" if its window-peak >= obs_peak - TOL

DATA_INNER_HOURS = 24   # 24 = daily; 6 = 6-hourly (needs High-RAM)
CHUNK_DAYS       = 5    # regrid-block size; peak RAM = one block at 0.25°

STJOHNS_LAT = 47.5615
STJOHNS_LON = 360.0 - 52.7126
RNG_SEED = 42

from pathlib import Path
DATA_DIR = Path("data"); PLOTS_DIR = Path("plots")
DATA_DIR.mkdir(exist_ok=True); PLOTS_DIR.mkdir(exist_ok=True)

## Load model, open ERA5, build regridder, load + regrid the window

In [ ]:
print(f"Loading model {MODEL_NAME} \u2026")
gcs = gcsfs.GCSFileSystem(token="anon")
with gcs.open(f"gs://neuralgcm/models/{MODEL_NAME}", "rb") as f:
    ckpt = pickle.load(f)
model = neuralgcm.PressureLevelModel.from_checkpoint(ckpt)
print("input_variables:", model.input_variables)

In [ ]:
print("Opening ARCO-ERA5 (lazy) \u2026")
full_era5 = xr.open_zarr(ERA5_PATH, chunks=None, storage_options=dict(token="anon"))

era5_grid = spherical_harmonic.Grid(
    latitude_nodes=full_era5.sizes["latitude"],
    longitude_nodes=full_era5.sizes["longitude"],
    latitude_spacing=xarray_utils.infer_latitude_spacing(full_era5.latitude),
    longitude_offset=xarray_utils.infer_longitude_offset(full_era5.longitude),
)
regridder = horizontal_interpolation.ConservativeRegridder(
    era5_grid, model.data_coords.horizontal, skipna=True)
shifted_full = full_era5[model.input_variables + model.forcing_variables].pipe(
    xarray_utils.selective_temporal_shift,
    variables=model.forcing_variables, time_shift="24 hours")


def load_regridded(t0, t1, inner_hours, chunk_days):
    sub = (shifted_full.sel(time=slice(t0, t1))
           .isel(time=slice(None, None, inner_hours)))
    steps = max(1, chunk_days * (24 // inner_hours))
    n = sub.sizes["time"]; pieces = []
    for i in range(0, n, steps):
        chunk = sub.isel(time=slice(i, i + steps)).compute()
        g = xarray_utils.fill_nan_with_nearest(xarray_utils.regrid(chunk, regridder))
        pieces.append(g)
        print(f"  block {i // steps + 1}: {chunk.sizes['time']:3d} snapshots regridded")
        del chunk; gc.collect()
    return xr.concat(pieces, dim="time")


buffer = np.timedelta64(2, "D")
window_start = EVENT_PEAK - np.timedelta64(max(LEAD_TIMES_DAYS), "D")
window_end   = EVENT_END + buffer
print(f"Loading + regridding ERA5 {window_start} \u2192 {window_end} \u2026")
eval_era5 = load_regridded(window_start, window_end, DATA_INNER_HOURS, CHUNK_DAYS)
print(f"\u2192 {eval_era5.sizes['time']} snapshots on the model grid.")


def daily_max_t1000(ds):
    pt = ds.temperature.sel(level=1000).sel(
        latitude=STJOHNS_LAT, longitude=STJOHNS_LON, method="nearest") - 273.15
    return pt.resample(time="1D").max().to_pandas()

era5_truth = daily_max_t1000(eval_era5)

## Ensemble sweep: N members per lead — batched with `jax.vmap` (different stochastic seeds, same IC)

In [ ]:
td = np.timedelta64(1, "h") * DATA_INNER_HOURS


def run_lead(init_time, keys):
    """Ensemble members for one lead, vmap'd over rng keys in GPU batches of
    MEMBER_BATCH. GPU memory ~ MEMBER_BATCH trajectories; HOST memory ~ ONE
    member (we keep the batch on-device and pull one member at a time, because
    each member's full model state is ~0.7 GB at 1.4° and materializing the
    whole batch on host crashes Colab RAM). Only the rng key varies; IC and
    forcings are shared (closed over)."""
    fc = eval_era5.sel(time=slice(init_time, window_end))
    n_steps = fc.sizes["time"]
    inputs       = model.inputs_from_xarray(fc.isel(time=0))
    forcing0     = model.forcings_from_xarray(fc.isel(time=0))
    all_forcings = model.forcings_from_xarray(fc)

    def one(key):
        state = model.encode(inputs, forcing0, key)        # stochastic seed = key
        _, preds = model.unroll(state, all_forcings, steps=n_steps,
                                timedelta=td, start_with_input=True)
        return preds

    batched_one = jax.vmap(one)
    times = fc.time.values
    members = []
    for s in range(0, keys.shape[0], MEMBER_BATCH):
        kb = keys[s:s + MEMBER_BATCH]
        preds_b = batched_one(kb)                          # batched, kept on device
        for j in range(kb.shape[0]):
            preds_m = jax.tree_util.tree_map(lambda x, i=j: np.asarray(x[i]), preds_b)
            ds = model.data_to_xarray(
                preds_m, times=np.arange(n_steps) * DATA_INNER_HOURS).as_numpy()
            ds = ds.assign_coords(time=times)
            members.append(daily_max_t1000(ds))
            del ds, preds_m; gc.collect()              # free this member before the next
        del preds_b; gc.collect()                      # free the device batch
    return members


base = jax.random.key(RNG_SEED)
win = slice(pd.Timestamp(EVENT_START), pd.Timestamp(EVENT_END))
obs_peak = float(era5_truth.loc[win].max())

ens_traj, records = {}, []
for lead in LEAD_TIMES_DAYS:
    init = EVENT_PEAK - np.timedelta64(lead, "D")
    keys = jax.random.split(jax.random.fold_in(base, int(lead)), N_MEMBERS)
    members = run_lead(init, keys)                         # batched in MEMBER_BATCH chunks
    mat = pd.concat(members, axis=1); mat.columns = range(N_MEMBERS)
    ens_traj[lead] = mat

    peaks = mat.loc[win].max(axis=0)                       # per-member window peak
    rmse = np.sqrt((mat.loc[win].sub(era5_truth.loc[win], axis=0) ** 2).mean(axis=0))
    hit = float((peaks >= obs_peak - CAPTURE_TOL).mean())
    records.append({"lead_days": lead, "init_date": pd.Timestamp(init).date(),
                    "hit_rate": round(hit, 2),
                    "ens_mean_peak_C": round(float(peaks.mean()), 2),
                    "ens_std_peak_C": round(float(peaks.std()), 2),
                    "obs_peak_C": round(obs_peak, 2),
                    "ens_mean_rmse_C": round(float(rmse.mean()), 2)})
    print(f"lead {lead:2d} d: hit {hit:.2f}  peak {peaks.mean():.1f}±{peaks.std():.1f} °C")
    gc.collect()

skill = pd.DataFrame(records).sort_values("lead_days")

# Spread sanity check: a deterministic checkpoint gives identical members.
chk = ens_traj[min(LEAD_TIMES_DAYS)]
if float(chk.std(axis=1).max()) < 1e-6:
    print("\n⚠️  ZERO ensemble spread — members are identical. You are likely using a "
          "DETERMINISTIC checkpoint; set MODEL_NAME to a stochastic one and re-run.")
skill

## Save outputs

In [ ]:
skill.to_csv(DATA_DIR / "heatwave_leadtime_ensemble_skill_aug2025.csv", index=False)
das = []
for lead, mat in ens_traj.items():
    da = xr.DataArray(mat.values, dims=("time", "member"),
                      coords={"time": mat.index, "member": np.arange(N_MEMBERS)}
                      ).expand_dims(lead_days=[lead])
    das.append(da)
ens_da = xr.concat(das, dim="lead_days")
xr.Dataset({"tmax_proxy_C": ens_da,
            "era5_tmax_proxy_C": xr.DataArray(
                era5_truth.values, coords={"time": era5_truth.index}, dims="time")}
          ).to_netcdf(DATA_DIR / "heatwave_leadtime_ensemble_aug2025.nc")
print("Saved \u2192 data/heatwave_leadtime_ensemble_skill_aug2025.csv + .nc")

## Plots

In [ ]:
obs = pd.Series({pd.Timestamp(k): v for k, v in OBS_TMAX.items()}).sort_index()
leads_desc = sorted(LEAD_TIMES_DAYS, reverse=True)
n = len(leads_desc); ncol = 5; nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 2.7 * nrow),
                         sharex=True, sharey=True)
axes = np.atleast_1d(axes).ravel()
for ax, lead in zip(axes, leads_desc):
    mat = ens_traj[lead]
    for col in mat.columns:
        ax.plot(mat.index, mat[col], color="C0", lw=0.6, alpha=0.4)
    ax.plot(mat.index, mat.mean(axis=1), color="C1", lw=1.8)
    ax.plot(era5_truth.index, era5_truth.values, "k", lw=1.4)
    ax.scatter(obs.index, obs.values, c="red", s=14, zorder=5)
    ax.axvspan(pd.Timestamp(EVENT_START), pd.Timestamp(EVENT_END), color="red", alpha=0.07)
    ax.set_title(f"init \u2212{lead} d")
    for lab in ax.get_xticklabels():
        lab.set_rotation(45); lab.set_ha("right")
for ax in axes[n:]:
    ax.axis("off")
fig.supylabel("daily-max T$_{1000}$ (\u00b0C)")
fig.suptitle("Ensemble forecasts by lead \u2014 members (blue), ens-mean (orange), "
             "ERA5 (black), station obs (red)", y=1.01)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "heatwave_leadtime_ensemble_envelope.pdf", bbox_inches="tight")
plt.show()

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
a1.plot(skill.lead_days, skill.hit_rate, "o-")
a1.set(xlabel="lead time (days before peak)", ylabel="fraction of members capturing",
       title=f"Hit rate vs lead  (capture: peak \u2265 {obs_peak - CAPTURE_TOL:.1f} \u00b0C)",
       ylim=(-0.05, 1.05))
a1.invert_xaxis(); a1.grid(alpha=0.3)

data = [ens_traj[l].loc[win].max(axis=0).values for l in leads_desc]
a2.boxplot(data, labels=[str(l) for l in leads_desc])
a2.axhline(obs_peak, color="red", ls="--", label=f"ERA5 peak {obs_peak:.1f} \u00b0C")
a2.set(xlabel="lead time (days before peak)",
       ylabel="member window-peak T$_{1000}$ (\u00b0C)",
       title="Per-member peak distribution vs lead")
a2.legend(); a2.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "heatwave_leadtime_ensemble_skill.pdf", bbox_inches="tight")
plt.show()
print(skill.to_string(index=False))

## Download outputs (Colab)

In [ ]:
try:
    from google.colab import files
    import shutil
    shutil.make_archive("heatwave_ensemble_outputs", "zip", ".", "data")
    shutil.make_archive("heatwave_ensemble_plots", "zip", ".", "plots")
    files.download("heatwave_ensemble_outputs.zip")
    files.download("heatwave_ensemble_plots.zip")
except Exception as e:
    print("Not on Colab / download skipped:", e)

## Reading the result

- **Hit rate vs lead is the headline.** Where it rises from ~0 toward ~1 is the robust
  crossover — the lead at which the heat wave is reliably in the forecast, not a single
  lucky run. Compare this crossover to the deterministic sweep's: if the deterministic
  run "caught" it earlier than the ensemble hit rate justifies, that catch was luck.
- **`CAPTURE_TOL` matters** — the hit rate depends on the capture threshold; try a couple
  of values (e.g. 1.5 / 2.0 / 3.0 °C) and report the sensitivity.
- **What this ensemble represents:** stochastic-physics spread from the same ERA5 IC
  (the NeuralGCM-idiomatic ensemble). It is *not* pure initial-condition (analysis-error)
  uncertainty. A complementary IC-perturbation ensemble — small perturbations to the input
  fields before `encode`, run with the deterministic model — would isolate IC sensitivity;
  ask if you want that variant too.
- **Next step:** the crossover lead time bounds the window in which the worst-case
  initial-condition optimisation is physically meaningful.